# 2 — Banded Vision Ensemble (ConvNets + attention, same threshold joining)

Same band protocol as notebook 1, applied to vision: a mixed pool of small ConvNets and
tiny ViTs trains independently, joins accuracy bands `[37..81..90]%` on a val split, observes
the grace period, then bag-distills *within* the band. Mixing architectures maximises
error independence (conv locality vs global attention).

**BuddyUp mapping:** `moderation_image` (NSFW MobileNetV3) and `food_recognition` heads;
form-pose frames as a second val domain.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(ai)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')
_missing = [m for m in ['torch', 'torchvision'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
# bootstrap.py reads BUDDY_SCALE at import time; if an earlier run in this
# same kernel cached the module (e.g. before the .env was loaded), drop the
# stale copy so the current environment is honoured.
for _stale in ('training.bootstrap', 'bootstrap'):
    _m = sys.modules.get(_stale)
    if _m is not None and getattr(_m, 'BUDDY_SCALE', None) != os.environ.get('BUDDY_SCALE'):
        sys.modules.pop(_stale, None)
        print(f'[bootstrap] re-importing {_stale} (stale scale cache cleared)')
try:
    from training.bootstrap import *
    CFG = init(scale=os.environ.get('BUDDY_SCALE') or None)
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

[env] /home/peter/Desktop/Buddy-Up/backend/.env | BUDDY_SCALE = demo | KAGGLE_API_TOKEN = set


2026-09-16 16:29:58.747496: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 16:29:58.904889: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 16:30:02.765146: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: demo


In [2]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(0); np.random.seed(0)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE = {'smoke': 1, 'demo': 2, 'full': 5}[SCALE]
N = {'smoke': 4, 'demo': 8, 'full': 20}[SCALE]

class SmallCNN(nn.Module):
    def __init__(self, c=10, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.f = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.h = nn.Linear(32, c)
    def forward(self, x): return self.h(self.f(x).flatten(1))

class TinyViT(nn.Module):
    """Patch-embed + 1 transformer layer + pool. Attention counterpart to the CNNs."""
    def __init__(self, c=10, d=64, patch=8, img=32, seed=0):
        super().__init__()
        self.proj = nn.Conv2d(3, d, patch, stride=patch)
        enc = nn.TransformerEncoderLayer(d, 4, dim_feedforward=128, batch_first=True)
        self.enc = enc; self.cls = nn.Parameter(torch.randn(d)); self.h = nn.Linear(d, c)
    def forward(self, x):
        t = self.proj(x).flatten(2).transpose(1, 2)
        t = torch.cat([self.cls.expand(len(x), 1, -1), t], 1)
        return self.h(self.enc(t)[:, 0])

class BandManager:
    def __init__(self, bands, grace): self.bands, self.grace, self.acc, self.joined = sorted(bands), grace, {}, {}
    def band_for(self, a):
        b = None
        for t in self.bands:
            if a >= t: b = t
        return b
    def update(self, mid, acc, ep):
        self.acc[mid] = max(acc, self.acc.get(mid, 0.))
        b = self.band_for(self.acc[mid])
        if b is not None and (mid not in self.joined or b > self.joined[mid][0]):
            self.joined[mid] = (b, ep); print(f'  [band] {mid} acc={self.acc[mid]:.3f} -> {int(b*100)}% @ep{ep}')
    def eligible(self, mid, ep): return mid in self.joined and (ep - self.joined[mid][1]) >= self.grace
    def pools(self):
        o = {}
        for m, (b, _) in self.joined.items(): o.setdefault(b, []).append(m)
        return o

In [3]:
# Synthetic striped-pattern images (stand-in). Real: CIFAR-10 / Food-101 / COCO / BuddyUp NSFW corpus.
def synth(n, img=32, c=10, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, 3, img, img)).astype(np.float32)
    y = rng.integers(0, c, n)
    for i in range(n): X[i, :, :, y[i]*3:(y[i]*3+3)] += 2.0  # class stripe
    return torch.tensor(X), torch.tensor(y)
Xtr, ytr = synth({'smoke': 1500, 'demo': 6000, 'full': 50000}[SCALE], seed=0)
Xva, yva = synth(1500, seed=1)
print('train/val:', tuple(Xtr.shape), tuple(Xva.shape))
# REAL DATA: torchvision.datasets.CIFAR10 / Food101, or ../data/nsfw/out/{train,val,test}.
# --- real batch (BUDDY_BATCH) overrides synthetic when present (uint8 -> float) ---
from batch_data import has_batch, batch_meta, load_tensors
if has_batch():
    _m = batch_meta(); _b = load_tensors('X', 'y', 'val_X', 'val_y')
    Xtr, ytr = _b['X'].float().div(255), _b['y']
    Xva, yva = _b['val_X'].float().div(255), _b['val_y']
    print('REAL batch train', tuple(Xtr.shape), 'val', tuple(Xva.shape), '|', _m['source'])


train/val: (6000, 3, 32, 32) (1500, 3, 32, 32)


REAL batch train (16638, 3, 32, 32) val (5546, 3, 32, 32) | data/nsfw/out


In [4]:
EPOCHS = {'smoke': 2, 'demo': 5, 'full': 20}[SCALE]
models = [(SmallCNN(seed=i) if i % 2 == 0 else TinyViT(seed=i)) for i in range(N)]
opts = [torch.optim.Adam(m.parameters(), lr=2e-3) for m in models]
mgr = BandManager(BANDS, GRACE); ce = nn.CrossEntropyLoss(); BS = 256
for ep in range(EPOCHS):
    for m, o in zip(models, opts):
        m.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i+BS]; o.zero_grad()
            loss = ce(m(Xtr[idx]), ytr[idx]); loss.backward(); o.step()
    print(f'--- epoch {ep} ---')
    with torch.no_grad():
        for j, m in enumerate(models):
            m.eval(); mgr.update(type(m).__name__+str(j), (m(Xva).argmax(1) == yva).float().mean().item(), ep)
    pools = mgr.pools()
    for b, members in pools.items():
        elig = [j for j, m in enumerate(models) if type(m).__name__+str(j) in members and mgr.eligible(type(m).__name__+str(j), ep)]
        if len(elig) < 2: continue
        with torch.no_grad():
            soft = torch.softmax(torch.stack([models[j](Xtr[:256]) for j in elig]).mean(0) / 2.0, -1)
        for j in elig:
            models[j].train(); opts[j].zero_grad()
            kd = -(soft * torch.log_softmax(models[j](Xtr[:256]) / 2.0, -1)).sum(-1).mean()
            (0.25 * kd).backward(); opts[j].step()
        print(f'  [distill] band {int(b*100)}%: distilled {len(elig)} members')
print('pools:', {int(k*100): v for k, v in mgr.pools().items()})

--- epoch 0 ---


  [band] SmallCNN0 acc=0.604 -> 56% @ep0


  [band] TinyViT1 acc=0.604 -> 56% @ep0


  [band] SmallCNN2 acc=0.604 -> 56% @ep0


  [band] TinyViT3 acc=0.607 -> 56% @ep0


...

  [band] TinyViT1 acc=0.671 -> 67% @ep3


  [band] TinyViT3 acc=0.676 -> 67% @ep3


  [distill] band 56%: distilled 5 members


--- epoch 4 ---


  [distill] band 56%: distilled 5 members
pools: {56: ['SmallCNN0', 'SmallCNN2', 'SmallCNN4', 'SmallCNN6', 'TinyViT7'], 67: ['TinyViT1', 'TinyViT3', 'TinyViT5']}


In [5]:
with torch.no_grad():
    accs = [(m(Xva).argmax(1) == yva).float().mean().item() for m in models]
    bag = (torch.stack([m(Xva) for m in models]).mean(0).argmax(1) == yva).float().mean().item()
print('members:', [round(a, 3) for a in accs])
print(f'best={max(accs):.3f} bagged={bag:.3f}')
torch.onnx.export(models[int(np.argmax(accs))].eval(), torch.randn(1, 3, 32, 32), '../models/banded_vision_best.onnx',
    input_names=['image'], output_names=['logits'], dynamic_axes={'image': {0: 'batch'}})
print('exported ../models/banded_vision_best.onnx')

members: [0.603, 0.68, 0.602, 0.673, 0.621, 0.703, 0.609, 0.611]
best=0.703 bagged=0.649


/tmp/ipykernel_2881284/2228477648.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(models[int(np.argmax(accs))].eval(), torch.randn(1, 3, 32, 32), '../models/banded_vision_best.onnx',


[torch.onnx] Obtain model graph for `TinyViT([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `TinyViT([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/banded_vision_best.onnx


## Data, scrapers vs platforms (notebook 2)

| Option | What to use | Verdict |
|---|---|---|
| Public datasets (start) | CIFAR-10/100, ImageNet-1k, COCO, OpenImages, Food-101 (`ethz/food101`) | Start here; Food-101 maps directly to BuddyUp food_recognition. |
| BuddyUp first-party | `data/nsfw/out/*` + user workout/form frames (consent-gated) | Best for moderation/form; needs DVC + label UI before scale. |
| Scrapers / bots | Common Crawl images, IG/TikTok scrapers | Avoid for people/fitness imagery — consent + copyright risk. Use LAION-5B / COYO (filtered CLIP data) instead of raw scraping. |
| Managed platforms | Roboflow (label+version), HuggingFace + Timm backbones, W&B sweeps, SageMaker/Vertex multi-node | No vendor offers banded-joining; run this notebook's loop as a sweep (one job per member, shared val ledger). |